Value obj is created to store from what values the present element exists
Calling .backward() computes gradients automatically

In [9]:
import math

class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op  # for graph/debug

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        out._backward = _backward
        return out

    def __sub__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data - other.data, (self, other), '-')

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += -1.0 * out.grad

        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __neg__(self):
        return self * -1

    def __radd__(self, other):
        return self + other

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad

        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()

        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)

        build(self)
        self.grad = 1.0

        for node in reversed(topo):
            node._backward()

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"


Neuron

In [2]:
import random

class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0.0)

    def __call__(self, x):
        # weighted sum + bias
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]


Layer

In [3]:
class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        return [n(x) for n in self.neurons]

    def parameters(self):
        params = []
        for neuron in self.neurons:
            params.extend(neuron.parameters())
        return params


MLP (Multi-Layer Perceptron)

In [4]:
class MLP:
    def __init__(self, nin, nouts):
        sizes = [nin] + nouts
        self.layers = [
            Layer(sizes[i], sizes[i+1])
            for i in range(len(nouts))
        ]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        params = []
        for layer in self.layers:
            params.extend(layer.parameters())
        return params


Loss = Squared error

In [10]:
def mse_loss(y_pred, y_true):
    return sum(
        ((yp - yt) * (yp - yt) for yp, yt in zip(y_pred, y_true)),
        Value(0.0)
    )


Training

In [11]:
# XOR dataset
X = [
    [Value(0.0), Value(0.0)],
    [Value(0.0), Value(1.0)],
    [Value(1.0), Value(0.0)],
    [Value(1.0), Value(1.0)],
]

Y = [
    [Value(0.0)],
    [Value(1.0)],
    [Value(1.0)],
    [Value(0.0)],
]

model = MLP(2, [8, 1])
lr = 0.1

for epoch in range(2000):
    # forward
    y_preds = [model(x) for x in X]
    loss = sum(mse_loss(yp, yt) for yp, yt in zip(y_preds, Y))

    # zero grads
    for p in model.parameters():
        p.grad = 0.0

    # backward
    loss.backward()

    # update
    for p in model.parameters():
        p.data -= lr * p.grad

    if epoch % 200 == 0:
        print(f"Epoch {epoch}, Loss: {loss.data}")


Epoch 0, Loss: 2.4087113672131246
Epoch 200, Loss: 0.07798430179597855
Epoch 400, Loss: 0.025967106298754948
Epoch 600, Loss: 0.016153672253686583
Epoch 800, Loss: 0.01171037928139963
Epoch 1000, Loss: 0.009176840582234674
Epoch 1200, Loss: 0.007540328737173502
Epoch 1400, Loss: 0.006396389500342396
Epoch 1600, Loss: 0.005551824653483941
Epoch 1800, Loss: 0.004902733044056803
